# PTB-XL ECG Signal Processing Pipeline

This notebook processes raw ECG signals from the PTB-XL database through a complete signal processing pipeline:
1. DC offset removal
2. Baseline wander removal (cascade median filter)
3. Notch filter (50/60 Hz powerline interference)
4. Bandpass filter (0.5-40 Hz)
5. Wavelet denoising

The processed signals are then saved to JSON for the web application.

In [ ]:
import pandas as pd
import wfdb
import numpy as np
import scipy.signal as sp_signal
import pywt
import matplotlib.pyplot as plt
import json
from pathlib import Path
from tqdm import tqdm

print("Libraries imported successfully!")

## Signal Processing Functions

In [ ]:
# -------------------------
# Utility / Filters
# -------------------------
def _ensure_odd(k):
    k = int(max(1, round(k)))
    return k if (k % 2 == 1) else k + 1

def moving_average(x, kernel_len):
    if kernel_len <= 1:
        return x.copy()
    kernel = np.ones(kernel_len, dtype=float) / float(kernel_len)
    return np.convolve(x, kernel, mode='same')

def notch_filter(sig, fs, freq=50.0, Q=30.0):
    """Zero-phase notch filter (returns same length)"""
    b, a = sp_signal.iirnotch(w0=freq, Q=Q, fs=fs)
    return sp_signal.filtfilt(b, a, sig)

def bandpass_filter(sig, fs, low=0.5, high=40.0, order=4):
    """Zero-phase Butterworth bandpass"""
    nyq = 0.5 * fs
    if low <= 0 or high >= nyq:
        raise ValueError("Invalid band limits for sampling freq")
    b, a = sp_signal.butter(order, [low / nyq, high / nyq], btype='band')
    return sp_signal.filtfilt(b, a, sig)

def wavelet_denoise(sig, wavelet='db6', level=None):
    """
    Soft-thresholding wavelet denoising that returns same-length signal.
    """
    maxlev = pywt.dwt_max_level(len(sig), pywt.Wavelet(wavelet).dec_len)
    if level is None:
        level = max(1, min(6, maxlev))

    coeffs = pywt.wavedec(sig, wavelet, level=level, mode='symmetric')
    detail_coeffs = coeffs[-1]
    sigma = np.median(np.abs(detail_coeffs)) / 0.6745 if detail_coeffs.size else 0.0
    uthresh = sigma * np.sqrt(2 * np.log(len(sig))) if sigma > 0 else 0.0

    coeffs_thresh = [coeffs[0]] + [pywt.threshold(c, value=uthresh, mode='soft') for c in coeffs[1:]]
    rec = pywt.waverec(coeffs_thresh, wavelet, mode='symmetric')

    if len(rec) > len(sig):
        rec = rec[:len(sig)]
    elif len(rec) < len(sig):
        rec = np.pad(rec, (0, len(sig) - len(rec)), mode='edge')
    return rec

print("Filter functions defined!")

In [ ]:
# -------------------------
# Baseline removal
# -------------------------
from scipy.signal import medfilt

def remove_baseline_wander(sig,
                           fs,
                           method='cascade',
                           hp_cutoff=0.5,
                           hp_order=2,
                           med_spike_ms=25,
                           med_baseline_ms=200,
                           smooth_baseline_ms=None,
                           return_baseline=False):
    """
    method: 'butter', 'median', 'cascade'
    returns corrected (and baseline if return_baseline=True)
    """
    sig = np.asarray(sig, dtype=float)
    n = len(sig)

    if method == 'butter':
        nyq = 0.5 * fs
        if not (0 < hp_cutoff < nyq):
            raise ValueError("hp_cutoff must be between 0 and Nyquist (fs/2)")
        b, a = sp_signal.butter(hp_order, hp_cutoff / nyq, btype='high', analog=False)
        corrected = sp_signal.filtfilt(b, a, sig, method="pad")
        if return_baseline:
            baseline = sig - corrected
            return corrected, baseline
        return corrected

    k_spike = _ensure_odd(int(round((med_spike_ms / 1000.0) * fs)))
    k_base  = _ensure_odd(int(round((med_baseline_ms / 1000.0) * fs)))

    if method == 'median':
        pad = k_base // 2
        padded = np.pad(sig, pad, mode='edge')
        baseline = medfilt(padded, kernel_size=k_base)[pad:pad + n]
        corrected = sig - baseline
        if return_baseline:
            return corrected, baseline
        return corrected

    # cascade method
    if k_spike <= 1:
        x1 = sig.copy()
    else:
        pad1 = k_spike // 2
        padded1 = np.pad(sig, pad1, mode='edge')
        x1 = medfilt(padded1, kernel_size=k_spike)[pad1:pad1 + n]

    pad2 = k_base // 2
    padded2 = np.pad(x1, pad2, mode='edge')
    baseline = medfilt(padded2, kernel_size=k_base)[pad2:pad2 + n]

    if smooth_baseline_ms is not None and smooth_baseline_ms > 0:
        k_smooth = int(round((smooth_baseline_ms / 1000.0) * fs))
        if k_smooth > 1:
            baseline = moving_average(baseline, k_smooth)

    corrected = sig - baseline
    if return_baseline:
        return corrected, baseline
    return corrected

print("Baseline removal function defined!")

In [ ]:
# -------------------------
# Pipeline runner
# -------------------------
def run_pipeline(ecg_raw, fs,
                 dc_remove='median',
                 baseline_params=None,
                 notch_params=None,
                 bandpass_params=None,
                 wavelet_params=None):
    """Run full pipeline, returns dict of intermediate signals."""
    out = {}
    out['raw'] = ecg_raw.copy()

    # DC removal
    if dc_remove == 'median':
        out['dc_removed'] = ecg_raw - np.median(ecg_raw)
    elif dc_remove == 'mean':
        out['dc_removed'] = ecg_raw - np.mean(ecg_raw)
    else:
        out['dc_removed'] = ecg_raw.copy()

    # Baseline wander
    if baseline_params is None:
        baseline_params = {}
    corrected, baseline = remove_baseline_wander(out['dc_removed'], fs, return_baseline=True, **baseline_params)
    out['baseline_estimate'] = baseline
    out['baseline_removed'] = corrected

    # Notch
    if notch_params is None:
        notch_params = {}
    out['notched'] = notch_filter(out['baseline_removed'], fs, **notch_params)

    # Bandpass
    if bandpass_params is None:
        bandpass_params = {}
    out['bandpassed'] = bandpass_filter(out['notched'], fs, **bandpass_params)

    # Wavelet denoise
    if wavelet_params is None:
        wavelet_params = {}
    out['denoised'] = wavelet_denoise(out['bandpassed'], **wavelet_params)

    return out

print("Pipeline function defined!")

## Load PTB-XL Database

In [ ]:
# Set your PTB-XL database path
BASE_PATH = r"C:\Users\Harsha\Pictures\RM\archive\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"

# Load database CSV
df = pd.read_csv(f"{BASE_PATH}/ptbxl_database.csv")
print(f"Loaded {len(df)} records from PTB-XL database")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst record:\n{df.iloc[0]}")

## Process Sample Record with Pipeline

In [ ]:
# Process one record as example
record_index = 0
record_info = df.iloc[record_index]
filename_hr = record_info['filename_hr']
full_path = f"{BASE_PATH}/{filename_hr}"

print(f"Loading record: {full_path}")
record = wfdb.rdrecord(full_path)

# Get Lead I signal
ecg_signal = record.p_signal[:, 0]  # Lead I
fs = int(record.fs)
t = np.linspace(0, len(ecg_signal) / fs, len(ecg_signal))

print(f"Signal length: {len(ecg_signal)} samples")
print(f"Sampling rate: {fs} Hz")
print(f"Duration: {len(ecg_signal) / fs:.2f} seconds")
print(f"Signal range: [{ecg_signal.min():.4f}, {ecg_signal.max():.4f}]")

In [ ]:
# Run the complete signal processing pipeline
baseline_params = {
    'method': 'cascade',
    'med_spike_ms': 25,
    'med_baseline_ms': 200,
    'smooth_baseline_ms': 100
}
notch_params = {'freq': 50.0, 'Q': 30.0}
bandpass_params = {'low': 0.5, 'high': 40.0, 'order': 4}
wavelet_params = {'wavelet': 'db6', 'level': 4}

results = run_pipeline(ecg_signal, fs,
                       dc_remove='median',
                       baseline_params=baseline_params,
                       notch_params=notch_params,
                       bandpass_params=bandpass_params,
                       wavelet_params=wavelet_params)

print("Pipeline processing complete!")
print(f"Available stages: {list(results.keys())}")

## Visualize Processing Stages

In [ ]:
plt.figure(figsize=(18, 14))

plt.subplot(6, 1, 1)
plt.plot(t, results['raw'], color='gray')
plt.title("Original ECG (Lead I)")
plt.grid()

plt.subplot(6, 1, 2)
plt.plot(t, results['dc_removed'], color='red')
plt.title("After DC Offset Removal")
plt.grid()

plt.subplot(6, 1, 3)
plt.plot(t, results['baseline_removed'], color='orange')
plt.plot(t, results['baseline_estimate'], color='k', linewidth=0.8, alpha=0.6, label='baseline estimate')
plt.legend()
plt.title("After Baseline Wander Removal (baseline est overplotted)")
plt.grid()

plt.subplot(6, 1, 4)
plt.plot(t, results['notched'], color='green')
plt.title("After Notch Filter (50 Hz)")
plt.grid()

plt.subplot(6, 1, 5)
plt.plot(t, results['bandpassed'], color='blue')
plt.title("After Bandpass Filter (0.5–40 Hz)")
plt.grid()

plt.subplot(6, 1, 6)
plt.plot(t, results['denoised'], color='purple')
plt.title("After Wavelet Denoising")
plt.xlabel("Time (s)")
plt.grid()

plt.tight_layout()
plt.show()

print("Processing stages visualized!")

## Process Multiple Records and Save for Web App

In [ ]:
def process_record_all_leads(record_info, base_path, num_samples=5000):
    """Process all 12 leads of a record through the pipeline"""
    filename_hr = record_info['filename_hr']
    full_path = f"{base_path}/{filename_hr}"
    
    try:
        record = wfdb.rdrecord(full_path)
        fs = int(record.fs)
        
        lead_names = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
        processed_leads = {}
        
        # Pipeline parameters
        baseline_params = {
            'method': 'cascade',
            'med_spike_ms': 25,
            'med_baseline_ms': 200,
            'smooth_baseline_ms': 100
        }
        notch_params = {'freq': 50.0, 'Q': 30.0}
        bandpass_params = {'low': 0.5, 'high': 40.0, 'order': 4}
        wavelet_params = {'wavelet': 'db6', 'level': 4}
        
        for i, lead_name in enumerate(lead_names):
            if i >= record.p_signal.shape[1]:
                break
                
            ecg_signal = record.p_signal[:, i]
            
            # Run pipeline
            results = run_pipeline(ecg_signal, fs,
                                 dc_remove='median',
                                 baseline_params=baseline_params,
                                 notch_params=notch_params,
                                 bandpass_params=bandpass_params,
                                 wavelet_params=wavelet_params)
            
            # Downsample for web display (keep first num_samples)
            final_signal = results['denoised'][:num_samples]
            
            processed_leads[lead_name] = {
                'raw': ecg_signal[:num_samples].tolist(),
                'processed': final_signal.tolist(),
                'dc_removed': results['dc_removed'][:num_samples].tolist(),
                'baseline_removed': results['baseline_removed'][:num_samples].tolist(),
                'notched': results['notched'][:num_samples].tolist(),
                'bandpassed': results['bandpassed'][:num_samples].tolist()
            }
        
        return processed_leads, fs
        
    except Exception as e:
        print(f"Error processing {full_path}: {e}")
        return None, None

print("Record processing function defined!")

In [ ]:
# Process first 50 records (adjust as needed)
num_records = 50
processed_records = []

for idx in tqdm(range(num_records), desc="Processing records"):
    record_info = df.iloc[idx]
    
    processed_leads, fs = process_record_all_leads(record_info, BASE_PATH, num_samples=5000)
    
    if processed_leads:
        record_data = {
            'ecg_id': int(record_info['ecg_id']),
            'patient_id': int(record_info['patient_id']),
            'age': int(record_info['age']) if pd.notna(record_info['age']) else 0,
            'sex': 'M' if record_info['sex'] == 1 else 'F',
            'sampling_rate': fs,
            'diagnosis': str(record_info['report']) if pd.notna(record_info['report']) else '',
            'scp_codes': str(record_info['scp_codes']) if pd.notna(record_info['scp_codes']) else '{}',
            'leads': processed_leads
        }
        processed_records.append(record_data)

print(f"\nProcessed {len(processed_records)} records successfully!")

In [ ]:
# Save to JSON file for web application
output_path = r"C:\Users\Harsha\Pictures\RM\public\ptbxl_processed_signals.json"

with open(output_path, 'w') as f:
    json.dump(processed_records, f, indent=2)

print(f"Saved processed signals to: {output_path}")
print(f"File size: {Path(output_path).stat().st_size / (1024*1024):.2f} MB")

## Verify Processed Data

In [ ]:
# Load and verify the saved data
with open(output_path, 'r') as f:
    loaded_data = json.load(f)

print(f"Loaded {len(loaded_data)} records")
print(f"\nFirst record structure:")
first_record = loaded_data[0]
print(f"ECG ID: {first_record['ecg_id']}")
print(f"Patient ID: {first_record['patient_id']}")
print(f"Age: {first_record['age']} | Sex: {first_record['sex']}")
print(f"Sampling rate: {first_record['sampling_rate']} Hz")
print(f"Available leads: {list(first_record['leads'].keys())}")
print(f"\nLead I data:")
lead_i = first_record['leads']['I']
print(f"  - Raw signal: {len(lead_i['raw'])} samples")
print(f"  - Processed signal: {len(lead_i['processed'])} samples")
print(f"  - Processing stages: {list(lead_i.keys())}")

In [ ]:
# Visualize one processed signal
lead_i_processed = np.array(first_record['leads']['I']['processed'])
t_display = np.linspace(0, len(lead_i_processed) / first_record['sampling_rate'], len(lead_i_processed))

plt.figure(figsize=(16, 4))
plt.plot(t_display, lead_i_processed, color='purple', linewidth=0.8)
plt.title(f"Processed ECG Signal - Lead I (Patient {first_record['patient_id']})")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (mV)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Signal statistics:")
print(f"  Mean: {lead_i_processed.mean():.4f}")
print(f"  Std: {lead_i_processed.std():.4f}")
print(f"  Min: {lead_i_processed.min():.4f}")
print(f"  Max: {lead_i_processed.max():.4f}")

## Complete! Ready for Web Application

The processed signals are now saved in `ptbxl_processed_signals.json` and ready to be loaded by the React web application. The signals have been:

1. ✅ DC offset removed
2. ✅ Baseline wander removed using cascade median filtering
3. ✅ Notch filtered (50 Hz powerline interference)
4. ✅ Bandpass filtered (0.5-40 Hz clinical ECG range)
5. ✅ Wavelet denoised for clean visualization

All 12 leads are included with multiple processing stages available for display.